[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/notebooks/RO/capitol4_notebook_seminar.ipynb)

# Modelarea piețelor financiare — Seminarul 4: Optimizarea portofoliului

*Notebook de seminar. Daniel Traian Pele, Antoaneta Amza — Academia de Studii Economice din București.*

- **Partea A**: verificarea numerică a celor nouă derivări.
- **Partea B**: backtest-uri și inferență pe date din Statele Unite ale Americii (SUA) și Bitcoin.
- **Partea C**: C1, critica unui răspuns Artificial Intelligence (AI); C2, un portofoliu optim de blue chips Bursa de Valori București (BVB) în Romanian leu (RON) (aduce diversificarea dincolo de BVB un cîștig?); C3, frontiera implementabilă pe datele cursului.
- Problemele **[Rezolvat]** sînt rezolvate complet (cod și rezultat); problemele **[Propus]** au o celulă goală pentru rezolvarea dumneavoastră: rezolvările se discută la seminar.
- Exemplul rezolvat de urmat: A1 pentru A9; A2 pentru A3; A4 pentru A5, A8; A6 pentru A7; B1 pentru B2, B9; B3 pentru B7; B4 pentru B8; A6, A7 pentru B5; B1, A9 pentru B6; B1, A4 pentru C1; B1, B4, A4 pentru C2.

> Rezolvările se discută la seminar.

In [1]:
import os
import io
import re
import json
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.spatial.distance import pdist
import warnings
warnings.filterwarnings('ignore')

Matplotlib is building the font cache; this may take a moment.


In [2]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
FRENCH = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'
BNR_XML = 'https://curs.bnr.ro/files/xml/years/nbrfxrates{}.xml'

SECTORS = ['XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY']        # from Dec 1998
SECTOR_NAMES = {'XLB': 'Materials', 'XLE': 'Energy', 'XLF': 'Financials', 'XLI': 'Industrials',
                'XLK': 'Technology', 'XLP': 'Cons. staples', 'XLU': 'Utilities', 'XLV': 'Health care',
                'XLY': 'Cons. discretionary'}
MULTI = ['SPY', 'EFA', 'EEM', 'TLT', 'IEF', 'LQD', 'HYG', 'GLD']                  # HYG from Apr 2007
MULTI_NAMES = {'SPY': 'US equities', 'EFA': 'Developed ex-US', 'EEM': 'Emerging markets',
               'TLT': 'US Treasuries 20y+', 'IEF': 'US Treasuries 7-10y', 'LQD': 'Investment-grade credit',
               'HYG': 'High-yield credit', 'GLD': 'Gold'}
BVB = ['TLV', 'SNP', 'BRD', 'TGN', 'SNG', 'SNN', 'EL', 'TEL']                      # listed before 2014-09
BVB_NAMES = {'TLV': 'Banca Transilvania', 'SNP': 'OMV Petrom', 'BRD': 'BRD-GSG', 'TGN': 'Transgaz',
             'SNG': 'Romgaz', 'SNN': 'Nuclearelectrica', 'EL': 'Electrica', 'TEL': 'Transelectrica'}

_CACHE = {}


def read_market(symbol):
    """Daily price table of one symbol (course data)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def price(symbol):
    """Price used: adjusted close for ETFs and stocks, close for indices and crypto."""
    d = read_market(symbol)
    col = 'close' if symbol.endswith('.INDX') or symbol.endswith('.CC') or symbol == 'BET' else 'adjusted_close'
    s = d[col].astype(float)
    return s[s > 0].rename(symbol)


def prices(symbols, start=None, end=None):
    """Prices aligned on common days (multi-asset analyses align prices first, then take returns)."""
    p = pd.concat([price(s) for s in symbols], axis=1).dropna()
    return p.loc[start:end]


def log_returns(p):
    """Log returns of a table already aligned on common days."""
    return np.log(p).diff().dropna()


def french_rf():
    """Monthly risk-free rate (one-month T-bill) from the Fama-French three-factor file, in decimals."""
    if 'rf' in _CACHE:
        return _CACHE['rf']
    url = FRENCH + 'F-F_Research_Data_Factors_CSV.zip'
    raw = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'}),
                                 timeout=120).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    text = z.read(z.namelist()[0]).decode('latin-1')
    rows = []
    for line in text.splitlines():
        parts = [x.strip() for x in line.split(',')]
        if len(parts) == 5 and len(parts[0]) == 6 and parts[0].isdigit():
            rows.append((parts[0], float(parts[4]) / 100.0))
        elif rows and not (len(parts[0]) == 6 and parts[0].isdigit()):
            break
    rf = pd.Series(dict(rows))
    rf.index = pd.to_datetime(rf.index, format='%Y%m') + pd.offsets.MonthEnd(0)
    _CACHE['rf'] = rf.rename('RF')
    return _CACHE['rf']


def monthly_returns(symbols, start=None):
    """Monthly simple returns from month-end prices (common days)."""
    p = prices(symbols).resample('ME').last()
    return p.pct_change().dropna().loc[start:]


def bnr_rate(currency='USD', start=2014, end=2026):
    """BNR reference rate: RON per unit of currency (annual XML archives)."""
    key = ('bnr', currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(start, end + 1):
        xml = urllib.request.urlopen(urllib.request.Request(BNR_XML.format(y), headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"( multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(2))))
    s = pd.Series(dict(rows)).sort_index()
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.rename(f'{currency}RON')
    return _CACHE[key]

In [3]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Chart colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
PALETTE = [MainBlue, IDAred, Forest, Amber, Orange, Purple, Crimson, '#795548', '#17A2B8', '#6F42C1', '#20C997']
EWcol    = '#D63384'   # colour of the 1/N portfolio (series are never grey)

SEED = 42
MAX_ABS_RET = 0.5        # threshold for data errors in the BVB series (daily log return)


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


US_END = '2026-07-31'
BVB_END = '2026-08-31'
WINDOW = 60
STRATS = ['1/N', 'MV', 'MV-LO', 'GMV', 'GMV-LO', 'GMV-LW', 'ERC', 'HRP']
BVB_STRATS = ['1/N', 'MV-LO', 'GMV-LO', 'GMV-LW', 'ERC', 'HRP']
SCOL = {'1/N': EWcol, 'MV': IDAred, 'MV-LO': Orange, 'GMV': Amber, 'GMV-LO': Purple,
        'GMV-LW': MainBlue, 'ERC': Forest, 'HRP': '#17A2B8'}
COMBINED = SECTORS + [a for a in MULTI if a != 'SPY']
UNIVERSES = {'Sectors': SECTORS, 'Multi-asset': MULTI, 'Combined': COMBINED}
LW_BLOCKS = (1, 2, 4, 6, 8, 10)   # Ledoit-Wolf (2008), Section 3.2.2
LW_K = 5000   # pseudo sequences for the block calibration (Algorithm 3.1)
LW_M = 4999   # bootstrap draws for the p-value
LW_M_CAL = 499   # bootstrap draws inside the calibration
LW_SB_MEAN = 5   # mean block of the stationary bootstrap of the VAR(1) residuals


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def us_monthly(symbols, start=None, end=US_END):
    """Monthly simple (total) returns and excess returns over RF, on common months."""
    r = monthly_returns([s + '.US' for s in symbols]).loc[start:end]
    r.columns = symbols
    rf = french_rf().reindex(r.index)
    ok = rf.notna()
    r, rf = r[ok], rf[ok]
    return r, r.sub(rf, axis=0), rf


def bvb_monthly(symbols=BVB, bench=('BETTR.INDX', 'BET'), end=BVB_END):
    """Monthly BVB returns (RON) from filtered daily returns (|r| > 0.5 = unadjusted corporate event)."""
    cols = [s + '.RO' for s in symbols] + list(bench)
    p = prices(cols)
    lr = log_returns(p)
    bad = lr.abs() > MAX_ABS_RET
    lr = lr.mask(bad, 0.0)
    m = np.exp(lr.resample('ME').sum()) - 1
    m = m.loc[:end].iloc[1:]                     # the first month is incomplete
    m.columns = list(symbols) + ['BET-TR' if b.startswith('BETTR') else b for b in bench]
    removed = {c.replace('.RO', ''): [str(d.date()) for d in lr.index[bad[c]]] for c in cols if bad[c].any()}
    return m, removed


def lw_cc(X):
    """Ledoit-Wolf (2004): shrinkage towards the constant-correlation target. Returns (Sigma, delta)."""
    X = np.asarray(X, float)
    T, N = X.shape
    Y = X - X.mean(0)
    S = Y.T @ Y / T
    var = np.diag(S).reshape(-1, 1)
    sd = np.sqrt(var)
    rbar = (np.sum(S / (sd @ sd.T)) - N) / (N * (N - 1))
    F = rbar * (sd @ sd.T)
    np.fill_diagonal(F, np.diag(S))
    Y2 = Y ** 2
    pi_mat = Y2.T @ Y2 / T - S ** 2
    pi_hat = pi_mat.sum()
    gamma_hat = np.linalg.norm(S - F, 'fro') ** 2
    theta = (Y ** 3).T @ Y / T - np.tile(var, (1, N)) * S
    np.fill_diagonal(theta, 0.0)
    rho_hat = np.trace(pi_mat) + rbar * np.sum(((1 / sd) @ sd.T) * theta)
    kappa = (pi_hat - rho_hat) / gamma_hat
    delta = max(0.0, min(1.0, kappa / T))
    return delta * F + (1 - delta) * S, delta


def w_ew(n):
    """The 1/N portfolio."""
    return np.ones(n) / n


def w_gmv(S):
    """Global minimum-variance portfolio: S^-1 1 / 1' S^-1 1."""
    x = np.linalg.solve(S, np.ones(len(S)))
    return x / x.sum()


def w_tan(mu, S):
    """Tangency (maximum-Sharpe) portfolio with short sales: S^-1 mu / 1' S^-1 mu."""
    x = np.linalg.solve(S, mu)
    return x / x.sum()


def w_long_only(S, mu=None):
    """Minimum variance (mu=None) or maximum Sharpe with weights 0 <= w <= 1 summing to 1 (SLSQP)."""
    n = len(S)
    if mu is None:
        f = lambda w: w @ S @ w
    else:
        f = lambda w: -(w @ mu) / np.sqrt(w @ S @ w)
    res = minimize(f, np.ones(n) / n, method='SLSQP', bounds=[(0, 1)] * n,
                   constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1}],
                   options={'ftol': 1e-12, 'maxiter': 500})
    w = np.clip(res.x, 0, None)
    return w / w.sum()


def w_erc(S):
    """Equal risk contributions (Maillard, Roncalli, Teiletche 2010): min 0.5 y'Sy - sum(log y)/n."""
    n = len(S)
    b = np.ones(n) / n
    f = lambda y: 0.5 * y @ S @ y - b @ np.log(y)
    grad = lambda y: S @ y - b / y
    y0 = 1 / np.sqrt(np.diag(S))
    res = minimize(f, y0 / y0.sum(), jac=grad, method='L-BFGS-B', bounds=[(1e-12, None)] * n,
                   options={'ftol': 1e-15, 'gtol': 1e-12, 'maxiter': 2000})
    return res.x / res.x.sum()


def hrp_tree(S):
    """HRP tree (Lopez de Prado 2016, Stage 1): d_ij = sqrt((1 - rho_ij)/2); then the Euclidean distance
    between the columns of D, d~_ij = sqrt(sum_n (d_ni - d_nj)^2); single linkage on d~."""
    sd = np.sqrt(np.diag(S))
    C = S / np.outer(sd, sd)
    D = np.sqrt(np.clip((1 - C) / 2, 0, None))
    np.fill_diagonal(D, 0)
    return linkage(pdist(D, 'euclidean'), 'single')


def w_hrp(S):
    """Hierarchical Risk Parity (Lopez de Prado 2016): tree ordering + recursive bisection."""
    S = np.asarray(S)
    order = list(leaves_list(hrp_tree(S)))
    w = np.ones(len(S))

    def cvar(idx):
        s = S[np.ix_(idx, idx)]
        iv = 1 / np.diag(s)
        iv /= iv.sum()
        return iv @ s @ iv

    clusters = [order]
    while clusters:
        nxt = []
        for c in clusters:
            if len(c) < 2:
                continue
            h = len(c) // 2
            a, b = c[:h], c[h:]
            va, vb = cvar(a), cvar(b)
            alpha = 1 - va / (va + vb)
            w[a] *= alpha
            w[b] *= 1 - alpha
            nxt += [a, b]
        clusters = nxt
    return w / w.sum()


def risk_contrib(w, S):
    """Percentage contributions to variance: w_i (S w)_i / w'Sw."""
    w = np.asarray(w)
    return w * (S @ w) / (w @ S @ w)


def weights(name, Rw):
    """Weights of strategy `name` estimated on the window Rw (excess returns, T x N)."""
    X = np.asarray(Rw, float)
    n = X.shape[1]
    mu = X.mean(0)
    S = np.cov(X.T)
    if name == '1/N':
        return w_ew(n)
    if name == 'MV':
        return w_tan(mu, S)
    if name == 'MV-LO':
        return w_long_only(S, mu)
    if name == 'GMV':
        return w_gmv(S)
    if name == 'GMV-LO':
        return w_long_only(S)
    if name == 'GMV-LW':
        return w_gmv(lw_cc(X)[0])
    if name == 'GMV-NL':
        return w_gmv(nl_shrink(X))
    if name == 'ERC':
        return w_erc(S)
    if name == 'HRP':
        return w_hrp(S)
    raise ValueError(name)


def backtest(R, Rex, window=WINDOW, strategies=STRATS):
    """Monthly rebalancing: weights from the last `window` months, applied in the next month.

    R: total returns, Rex: excess returns (same index). Returns the portfolios' excess returns,
    the turnover (sum |w_new - w_drifted|) and the weight history.
    """
    R, Rex = R.values, Rex
    idx = Rex.index[window:]
    X = Rex.values
    out = {s: [] for s in strategies}
    turn = {s: [] for s in strategies}
    W = {s: [] for s in strategies}
    prev = {s: None for s in strategies}
    for t in range(window, len(X)):
        est = X[t - window:t]
        for s in strategies:
            w = weights(s, est)
            if prev[s] is not None:
                turn[s].append(np.abs(w - prev[s]).sum())
            else:
                turn[s].append(np.nan)
            out[s].append(w @ X[t])
            gross = 1 + R[t]
            prev[s] = w * gross / (w @ gross)
            W[s].append(w)
    ret = pd.DataFrame(out, index=idx)
    ret.attrs['rf'] = pd.Series(R[window:, 0] - X[window:, 0], index=idx)   # risk-free rate (0 if R = Rex)
    to = pd.DataFrame(turn, index=idx)
    Wd = {s: pd.DataFrame(np.array(W[s]), index=idx, columns=Rex.columns) for s in strategies}
    return ret, to, Wd


def sharpe(r, periods=12):
    r = np.asarray(r)
    return r.mean() / r.std(ddof=1) * np.sqrt(periods)


def summary(ret, to, costs=(0, 10, 50)):
    """Mean, volatility, Sharpe (annualised, excess returns), mean monthly turnover, maximum drawdown
    of wealth from total returns (excess + risk-free rate), Sharpe net of costs (bp)."""
    rows = {}
    rf = ret.attrs.get('rf', 0.0)
    for s in ret.columns:
        r = ret[s]
        d = dict(mean=r.mean() * 12, vol=r.std() * np.sqrt(12), sharpe=sharpe(r),
                 turnover=to[s].mean(), mdd=max_drawdown(r + rf))
        for c in costs[1:]:
            d[f'sharpe_{c}bp'] = sharpe(r - c / 1e4 * to[s].fillna(0))
        rows[s] = d
    return pd.DataFrame(rows).T


def max_drawdown(r):
    """Maximum drawdown of wealth; a monthly loss >= 100% wipes out wealth (drawdown -100%)."""
    r = np.asarray(r, float)
    if (r <= -1).any():
        return -1.0
    w = np.r_[1.0, np.cumprod(1 + r)]                          # the initial wealth of 1 enters the running maximum
    return float((w / np.maximum.accumulate(w) - 1).min())


def sr_diff_hac(r1, r2, lags=None):
    """Ledoit-Wolf (2008), Section 3.1: Delta = SR1 - SR2 (monthly), delta-method standard error,
    HAC covariance with the Bartlett (Newey-West) kernel and lag floor(4 (T/100)^(2/9))."""
    r1, r2 = np.asarray(r1, float), np.asarray(r2, float)
    T = len(r1)
    m1, m2 = r1.mean(), r2.mean()
    g1, g2 = (r1 ** 2).mean(), (r2 ** 2).mean()
    d = m1 / np.sqrt(g1 - m1 ** 2) - m2 / np.sqrt(g2 - m2 ** 2)
    grad = np.array([g1 / (g1 - m1 ** 2) ** 1.5, -g2 / (g2 - m2 ** 2) ** 1.5,
                     -m1 / (2 * (g1 - m1 ** 2) ** 1.5), m2 / (2 * (g2 - m2 ** 2) ** 1.5)])
    Y = np.column_stack([r1 - m1, r2 - m2, r1 ** 2 - g1, r2 ** 2 - g2])
    if lags is None:
        lags = int(np.floor(4 * (T / 100) ** (2 / 9)))
    Psi = Y.T @ Y / T
    for l in range(1, lags + 1):
        G = Y[l:].T @ Y[:-l] / T
        Psi += (1 - l / (lags + 1)) * (G + G.T)
    se = np.sqrt(grad @ Psi @ grad / T)
    p = 2 * (1 - stats.norm.cdf(abs(d / se)))
    return d * np.sqrt(12), se * np.sqrt(12), p


def _lw_parts(r1, r2):
    """Delta = SR1 - SR2 (monthly), the gradient of f and the series y_t (LW 2008, eqs. 1-5)."""
    m1, m2 = r1.mean(-1), r2.mean(-1)
    g1, g2 = (r1 ** 2).mean(-1), (r2 ** 2).mean(-1)
    v1, v2 = g1 - m1 ** 2, g2 - m2 ** 2
    d = m1 / np.sqrt(v1) - m2 / np.sqrt(v2)
    grad = np.stack([g1 / v1 ** 1.5, -g2 / v2 ** 1.5, -m1 / (2 * v1 ** 1.5), m2 / (2 * v2 ** 1.5)], -1)
    Y = np.stack([r1 - m1[..., None], r2 - m2[..., None], r1 ** 2 - g1[..., None], r2 ** 2 - g2[..., None]], -1)
    return d, grad, Y


def qs_kernel(x):
    """Quadratic Spectral kernel (Andrews 1991)."""
    x = np.asarray(x, float)
    out = np.ones_like(x)
    nz = x != 0
    z = 6 * np.pi * x[nz] / 5
    out[nz] = 25 / (12 * np.pi ** 2 * x[nz] ** 2) * (np.sin(z) / z - np.cos(z))
    return out


def hac_qs_pw(Y):
    """HAC covariance with the prewhitened QS kernel (Andrews & Monahan 1992), automatic bandwidth
    (Andrews 1991, AR(1)), factor T/(T-4) of LW (2008, eq. 5). Y: T x 4, centred."""
    T, k = Y.shape
    X0, X1 = Y[:-1], Y[1:]
    A = np.linalg.lstsq(X0, X1, rcond=None)[0].T                # VAR(1) without intercept
    U, s, Vt = np.linalg.svd(A)                                  # Andrews-Monahan: singular values <= 0.97
    A = U @ np.diag(np.minimum(s, 0.97)) @ Vt
    E = X1 - X0 @ A.T
    n = len(E)
    rho = np.array([np.sum(E[1:, a] * E[:-1, a]) / np.sum(E[:-1, a] ** 2) for a in range(k)])
    sig2 = np.array([np.mean((E[1:, a] - rho[a] * E[:-1, a]) ** 2) for a in range(k)])
    num = np.sum(4 * rho ** 2 * sig2 ** 2 / (1 - rho) ** 8)
    den = np.sum(sig2 ** 2 / (1 - rho) ** 4)
    ST = 1.3221 * (num / den * n) ** 0.2
    Psi = E.T @ E / n
    for j in range(1, n):
        w = qs_kernel(j / ST)
        G = E[j:].T @ E[:-j] / n
        Psi += w * (G + G.T)
    B = np.linalg.inv(np.eye(k) - A)
    return T / (T - 4) * B @ Psi @ B.T


def lw_se(r1, r2):
    """Delta (monthly) and its standard error s(Delta) from the original data (LW 2008, eq. 5)."""
    d, grad, Y = _lw_parts(np.asarray(r1, float), np.asarray(r2, float))
    Psi = hac_qs_pw(Y)
    return float(d), float(np.sqrt(grad @ Psi @ grad / len(r1)))


def cbb_stat(r1, r2, b, M, rng):
    """M circular block bootstrap draws with block length b: Delta* and the 'natural' s(Delta*)
    (Goetze & Kuensch 1996; LW 2008, Section 3.2.2)."""
    T = len(r1)
    nb = -(-T // b)
    st = rng.integers(0, T, (M, nb))
    idx = ((st[:, :, None] + np.arange(b)) % T).reshape(M, -1)[:, :T]
    x1, x2 = r1[idx], r2[idx]
    d, grad, Y = _lw_parts(x1, x2)
    l = T // b
    Z = Y[:, :l * b].reshape(M, l, b, 4).sum(2) / np.sqrt(b)
    Psi = np.einsum('mjk,mjl->mkl', Z, Z) / l
    se = np.sqrt(np.einsum('mk,mkl,ml->m', grad, Psi, grad) / T)
    return d, se


def _sb_indices(T, n, mean_block, rng):
    """Indices of a stationary bootstrap (Politis & Romano 1994), mean block `mean_block`."""
    idx = np.empty(n, int)
    idx[0] = rng.integers(T)
    new = rng.random(n) < 1 / mean_block
    starts = rng.integers(0, T, n)
    for t in range(1, n):
        idx[t] = starts[t] if new[t] else (idx[t - 1] + 1) % T
    return idx


def lw_block_calibration(r1, r2, blocks=LW_BLOCKS, K=LW_K, M=LW_M_CAL, alpha=0.05, seed=SEED):
    """Algorithm 3.1 of LW (2008): VAR(1) + stationary bootstrap of the residuals, K pseudo sequences,
    coverage of the symmetric studentized interval for each b; picks the b whose coverage is closest to 1-alpha."""
    rng = np.random.default_rng(seed)
    r1, r2 = np.asarray(r1, float), np.asarray(r2, float)
    X = np.column_stack([r1, r2])
    T = len(X)
    d0, _ = lw_se(r1, r2)
    Z = np.column_stack([np.ones(T - 1), X[:-1]])
    coef = np.linalg.lstsq(Z, X[1:], rcond=None)[0]
    U = X[1:] - Z @ coef
    U = U - U.mean(0)
    cover = np.zeros(len(blocks))
    for k in range(K):
        u = U[_sb_indices(len(U), T - 1, LW_SB_MEAN, rng)]
        Xs = np.empty_like(X)
        Xs[0] = X[0]
        for t in range(1, T):
            Xs[t] = coef[0] + Xs[t - 1] @ coef[1:] + u[t - 1]
        dk, sk = lw_se(Xs[:, 0], Xs[:, 1])
        for j, b in enumerate(blocks):
            ds, ss = cbb_stat(Xs[:, 0], Xs[:, 1], b, M, rng)
            z = np.quantile(np.abs(ds - dk) / ss, 1 - alpha)
            cover[j] += abs(dk - d0) <= z * sk
    g_hat = cover / K
    return blocks[int(np.argmin(np.abs(g_hat - (1 - alpha))))], dict(zip(map(str, blocks), g_hat.tolist()))


def sr_diff_boot(r1, r2, block=None, M=LW_M, K=LW_K, alpha=0.05, seed=SEED):
    """Ledoit-Wolf (2008, Section 3.2.2 and Remark 3.2) test of H0: SR1 = SR2:
    studentized circular block bootstrap; block from Algorithm 3.1 (if block=None).
    Returns Delta (annualised), the symmetric 95% interval (annualised), the p-value (eq. 9), the
    bootstrap studentized statistics, the chosen block and the calibration output."""
    r1, r2 = np.asarray(r1, float), np.asarray(r2, float)
    cal = None
    if block is None:
        block, cal = lw_block_calibration(r1, r2, K=K, alpha=alpha, seed=seed)
    d, s = lw_se(r1, r2)
    ds, ss = cbb_stat(r1, r2, block, M, np.random.default_rng(seed + 1))
    tstar = (ds - d) / ss
    z = np.quantile(np.abs(tstar), 1 - alpha)
    p = (np.sum(np.abs(tstar) >= abs(d) / s) + 1) / (M + 1)
    a = np.sqrt(12)
    return dict(diff=d * a, se=s * a, ci=[(d - z * s) * a, (d + z * s) * a], p_boot=float(p),
                t_obs=d / s, z_star=float(z), block=int(block), calibration=cal), tstar


def _test_pair(args):
    """One Sharpe-difference test: HAC (Section 3.1) and studentized bootstrap (Section 3.2.2)."""
    r1, r2, block = args
    d, se, p = sr_diff_hac(r1, r2)
    res, tstar = sr_diff_boot(r1, r2, block=block)
    return dict(diff=d, se_hac=se, p_hac=p, ci_boot=res['ci'], p_boot=res['p_boot'], block=res['block'],
                calibration=res['calibration'], se_qs=res['se'], t_obs=res['t_obs'], z_star=res['z_star']), tstar


def run_tests(pairs, n_jobs=1, blocks=None):
    """pairs: dict {key: (r1, r2)}; blocks: dict {key: block} already calibrated (otherwise Algorithm 3.1);
    n_jobs > 1 uses parallel processes (the calibration is expensive)."""
    keys = list(pairs)
    blocks = blocks or {}
    args = [(np.asarray(pairs[k][0], float), np.asarray(pairs[k][1], float), blocks.get(k)) for k in keys]
    if n_jobs > 1:
        from concurrent.futures import ProcessPoolExecutor
        with ProcessPoolExecutor(n_jobs) as ex:
            res = list(ex.map(_test_pair, args))
    else:
        res = [_test_pair(a) for a in args]
    return dict(zip(keys, res))


def nl_shrink(X):
    """Analytical nonlinear shrinkage (Ledoit & Wolf 2020, Annals of Statistics), case N <= T-1."""
    X = np.asarray(X, float)
    X = X - X.mean(0)
    n, p = X.shape
    n = n - 1                                               # correction for centring
    S = X.T @ X / n
    lam, U = np.linalg.eigh(S)
    lam = np.maximum(lam, 1e-18)
    L = np.tile(lam[:, None], (1, p))
    h = n ** (-1 / 3)
    H = h * L.T
    x = (L - L.T) / H
    ft = (3 / 4 / np.sqrt(5)) * np.mean(np.maximum(1 - x ** 2 / 5, 0) / H, axis=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        Hf = (-3 / 10 / np.pi) * x + (3 / 4 / np.sqrt(5) / np.pi) * (1 - x ** 2 / 5) * \
            np.log(np.abs((np.sqrt(5) - x) / (np.sqrt(5) + x)))
    edge = np.isclose(np.abs(x), np.sqrt(5))
    Hf[edge] = (-3 / 10 / np.pi) * x[edge]
    Hft = np.mean(Hf / H, axis=1)
    c = p / n
    d = lam / ((np.pi * c * lam * ft) ** 2 + (1 - c - np.pi * c * lam * Hft) ** 2)
    return U @ np.diag(d) @ U.T


def to_py(o):
    if isinstance(o, dict):
        return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_py(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return to_py(o.tolist())
    if isinstance(o, pd.DataFrame):
        return to_py(o.to_dict(orient='index'))
    return o


def save_fig(name):
    """Afișează figura."""
    plt.show()
    plt.close()

In [4]:
SUMCOLS = {'mean': 'Mean', 'vol': 'Volatility', 'sharpe': 'Sharpe', 'turnover': 'Turnover', 'mdd': 'Max drawdown', 'sharpe_10bp': 'Sharpe net 10 bp', 'sharpe_50bp': 'Sharpe net 50 bp'}
def two_asset_stats():
    R, Rex, rf = us_monthly(['SPY', 'TLT'], start='2002-08-01')
    mu = R.mean() * 12
    sd = R.std() * np.sqrt(12)
    rho = R.corr().iloc[0, 1]
    return R, mu, sd, rho


def gmv_two(s1, s2, rho):
    """Weight of the first asset in the two-asset GMV portfolio."""
    return (s2 ** 2 - rho * s1 * s2) / (s1 ** 2 + s2 ** 2 - 2 * rho * s1 * s2)


def run_backtests():
    out = {}
    for name, syms in UNIVERSES.items():
        R, Rex, rf = us_monthly(syms)
        out[name] = backtest(R, Rex)
        print(f'   backtest {name}: {len(out[name][0])} months')
    return out


def sr_diff_iid(sr1, sr2, rho, T):
    """Sharpe ratio difference under i.i.d. Normal returns (formula reported by Ledoit-Wolf 2008).

    sr1, sr2: monthly Sharpe ratios; rho: correlation of the returns; returns (z, p, se).
    """
    v = (2 - 2 * rho + 0.5 * (sr1 ** 2 + sr2 ** 2 - 2 * sr1 * sr2 * rho ** 2)) / T
    z = (sr1 - sr2) / np.sqrt(v)
    return z, 2 * (1 - stats.norm.cdf(abs(z))), np.sqrt(v)


def var_ratio_boot(r1, r2, block=6, B=5000, seed=SEED, return_draws=False):
    """Out-of-sample variance ratio var(r1)/var(r2), with a circular block bootstrap interval.
    Both series use the SAME indices (paired months), so their correlation is preserved."""
    rng = np.random.default_rng(seed)
    r1, r2 = np.asarray(r1, float), np.asarray(r2, float)
    T = len(r1)
    nb = int(np.ceil(T / block))
    e1, e2 = np.r_[r1, r1[:block]], np.r_[r2, r2[:block]]
    out = np.empty(B)
    for b in range(B):
        st = rng.integers(0, T, nb)
        idx = (st[:, None] + np.arange(block)).ravel()[:T]
        out[b] = e1[idx].var() / e2[idx].var()
    if return_draws:
        return r1.var() / r2.var(), np.percentile(out, [2.5, 97.5]), np.mean(out >= 1), out
    return r1.var() / r2.var(), np.percentile(out, [2.5, 97.5]), np.mean(out >= 1)


def mp_bounds(c, s2=1.0):
    """Support of the Marchenko-Pastur law for the ratio c = N/T < 1."""
    return s2 * (1 - np.sqrt(c)) ** 2, s2 * (1 + np.sqrt(c)) ** 2


FOREIGN = ['SPY', 'EFA', 'IEF', 'GLD']
# backtest-urile din curs, refolosite mai jos
bts = run_backtests()

   backtest Sectors: 271 months


   backtest Multi-asset: 171 months


   backtest Combined: 171 months


## Partea A — exerciții pe hîrtie (verificare numerică)

- Celulele de mai jos verifică cifrele din A1–A9; enunțurile complete (ipoteze, cerințe, ce se raportează) sînt în slide-urile Seminarului 4.

### A1. Portofoliul Global Minimum-Variance portfolio (GMV) pentru N active [Rezolvat]

In [5]:
# A1 GMV cu două active: SPY și TLT
R, mu, sd, rho = two_asset_stats()
w = gmv_two(sd['SPY'], sd['TLT'], rho)
vol = np.sqrt(w**2 * sd['SPY']**2 + (1 - w)**2 * sd['TLT']**2 + 2 * w * (1 - w) * rho * sd['SPY'] * sd['TLT'])
print(f"sigma1 = {sd['SPY']:.4f}  sigma2 = {sd['TLT']:.4f}  rho = {rho:.4f}")
print(f"w* = {w:.3f}  vol = {vol:.4f}  medie = {w * mu['SPY'] + (1 - w) * mu['TLT']:.4f}")
Sig2 = np.array([[sd['SPY']**2, rho * sd['SPY'] * sd['TLT']], [rho * sd['SPY'] * sd['TLT'], sd['TLT']**2]])
A = np.ones(2) @ np.linalg.solve(Sig2, np.ones(2))
print(f'A = {A:.1f}  1/sqrt(A) = {1 / np.sqrt(A):.4f}  w_GMV = {(np.linalg.solve(Sig2, np.ones(2)) / A).round(3)}')

sigma1 = 0.1472  sigma2 = 0.1349  rho = -0.1005
w* = 0.461  vol = 0.0943  medie = 0.0773
A = 112.4  1/sqrt(A) = 0.0943  w_GMV = [0.461 0.539]


### A2. Equal Risk Contribution portfolio (ERC) cu două active [Rezolvat]

In [6]:
# A2 ERC cu două active = inversul volatilității
r, rex, rf = us_monthly(['SPY', 'IEF'])
S2 = r.cov().values * 12
s = np.sqrt(np.diag(S2))
w_iv = (1 / s) / (1 / s).sum()
print('inversul volatilității:', w_iv.round(3), ' ERC numeric:', w_erc(S2).round(3))
print('cote de risc:', risk_contrib(w_iv, S2).round(3))

inversul volatilității: [0.311 0.689]  ERC numeric: [0.311 0.689]
cote de risc: [0.5 0.5]


### A3. Repartizarea riscului în portofoliul 60/40 [Propus]

- **Model**: [A2](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A2) \[Rezolvat\]

- **Întrebare**: un portofoliu 60/40, considerat echilibrat, deține 40% în obligațiuni; ce parte din riscul lui provine din obligațiuni?

- **Date de intrare**: $\boldsymbol\Sigma$ din [A2](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A2); $\mathbf{w} = (0{,}6;\ 0{,}4)$ în SPY și iShares 7–10 Year Treasury Bond ETF (IEF)

- **Cerințe**

  1.  Calculați $\boldsymbol\Sigma\mathbf{w}$ rînd cu rînd: $(\boldsymbol\Sigma\mathbf{w})_1 = 0{,}6\sigma_1^2 + 0{,}4\sigma_{12}$ și $(\boldsymbol\Sigma\mathbf{w})_2 = 0{,}6\sigma_{12} + 0{,}4\sigma_2^2$.

  2.  Calculați $\sigma_p^2 = \mathbf{w}^\top(\boldsymbol\Sigma\mathbf{w})$, $\sigma_p$ și cotele $w_i(\boldsymbol\Sigma\mathbf{w})_i/\sigma_p^2$.

  3.  Comparați $\sigma_p$ cu volatilitatea ERC din [A2](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A2) și reprezentați alăturat cotele de capital și de risc ale celor două portofolii.

- **Interpretare**: este 60/40 un portofoliu echilibrat din punctul de vedere al riscului?

[Exercițiul A3 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A3)

In [7]:
# Codul vostru aici


### A4. Varianța unei diferențe de Sharpe [Rezolvat]

In [8]:
# A4 testul i.i.d. al diferenței de Sharpe
ret = bts['Sectors'][0]
sr1, sr2 = ret['GMV'].mean() / ret['GMV'].std(), ret['1/N'].mean() / ret['1/N'].std()
rho = ret[['GMV', '1/N']].corr().iloc[0, 1]
z, p, se = sr_diff_iid(sr1, sr2, rho, len(ret))
print(f'T = {len(ret)}  SR = {sr1:.4f}, {sr2:.4f}  rho = {rho:.4f}  se = {se:.4f}  z = {z:.2f}  p = {p:.3f}')
print('HAC p =', round(sr_diff_hac(ret['GMV'], ret['1/N'])[2], 3))

T = 271  SR = 0.2051, 0.1908  rho = 0.6939  se = 0.0483  z = 0.30  p = 0.766
HAC p = 0.754


### A5. ERC față de 1/N [Propus]

- Replicare: deduceți independent statistica și decizia, apoi comparați cu reperul $z=1{,}47$ din ilustrația testului bilateral.

- **Model**: [A4](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A4) \[Rezolvat\]

- **Întrebare**: avantajul ERC față de 1/N este mai mic decît al GMV; poate o diferență mai mică să fie mai aproape de semnificație?

- **Date de intrare**: backtest-ul [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B1) cu ERC adăugat, $T = 271$; Sharpe lunar $SR_{\text{ERC}} = 0{,}2018$, $SR_{1/N} = 0{,}1908$, corelația $0{,}9927$; păstrați patru zecimale pînă la final (paranteza este mică atunci cînd $\rho$ este aproape de 1)

- Reper: formula pentru $T\operatorname{Var}(\hat\Delta)$ din [[Varianța unei diferențe de Sharpe]{style="color: MainBlue"}](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=sem4p14)

- **Cerințe**

  1.  Calculați paranteza, eroarea standard, $z$ și p-value-ul bilateral; decideți la $5\%$.

  2.  Arătați că paranteza tinde la $\tfrac12(SR_1 - SR_2)^2$ cînd $\rho \to 1$.

  3.  Completați un tabel comparativ (diferență, $\rho$, eroare standard, $z$) pentru [A4](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A4) și [A5](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A5) și reprezentați ambele intervale de 95%.

- **Interpretare**

  - de ce este aici o diferență mai mică mai aproape de semnificație?

  - ce se întîmplă dacă cele două serii de randamente sînt identice?

[Exercițiul A5 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A5)

In [9]:
# Codul vostru aici


### A6. Black–Litterman cu un activ [Rezolvat]

In [10]:
# A6 medie ponderată cu precizia
pi, q, tau, sig, om = 0.05, 0.08, 0.05, 0.16, 0.02**2
pp, pv = 1 / (tau * sig**2), 1 / om
print(f'precizii {pp:.0f} {pv:.0f};  mu_BL = {(pp * pi + pv * q) / (pp + pv):.4f};  '
      f'ponderea opiniei = {pv / (pp + pv):.3f};  sd = {np.sqrt(1 / (pp + pv)):.4f}')

precizii 781 2500;  mu_BL = 0.0729;  ponderea opiniei = 0.762;  sd = 0.0175


### A7. Opinia sectorială din curs [Propus]

- Replicare: calcul/cod reproductibil, comparație cu reperul din curs și interpretare.

- **Model**: [A6](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A6) \[Rezolvat\]

- **Date de intrare**: nouă sectoare în ordinea XLB, XLE, XLF, XLI, XLK, XLP, XLU, XLV, XLY; randamente lunare în exces august 2016 -- iulie 2026, $\boldsymbol\Sigma$ anualizată; reper 1/N, aversiunea la risc $\delta_{\text{RA}} = 2{,}5$, $\tau = 0{,}05$; $\Delta\mathbf{w}$: modificarea ponderilor față de reper

- Opinia XLK minus XLU $= 3\%$ pe an: $\mathbf{P} = (0, 0, 0, 0, +1, 0, -1, 0, 0)$; implicit $\mathbf{P}\boldsymbol\pi = 2{,}22\%$; varianța spread-ului $\mathbf{P}\boldsymbol\Sigma\mathbf{P}^\top = 0{,}0477$ (suficientă; întreaga $\boldsymbol\Sigma$ este necesară doar pentru verificare)

- **Cerințe**

  1.  Scrieți distribuția a priori a lui $\mathbf{P}\boldsymbol\mu$ implicată de $\boldsymbol\mu \sim N(\boldsymbol\pi, \tau\boldsymbol\Sigma)$.

  2.  Cu $\Omega = \tau\,\mathbf{P}\boldsymbol\Sigma\mathbf{P}^\top$ ([He & Litterman (2002)](https://doi.org/10.2139/ssrn.334304)), arătați că opinia a posteriori este media simplă dintre a priori și opinie și calculați-o.

  3.  Arătați că $\Delta\mathbf{w} = \mathbf{P}^\top(q - \mathbf{P}\boldsymbol\pi)/(2\delta_{\text{RA}}\mathbf{P}\boldsymbol\Sigma\mathbf{P}^\top)$; calculați scalarul, noile ponderi XLK și XLU și verificați că ajustarea ponderilor are suma zero.

- **Interpretare**: cum se modifică ajustarea ponderilor dacă dispersia opiniei $\Omega$ se dublează (precizia se înjumătățește)?

[Exercițiul A7 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A7)

In [11]:
# Codul vostru aici


### A8. Deplasarea raportului Sharpe în eșantion [Propus]

- Replicare: calcul/cod reproductibil, comparație cu reperul din curs și interpretare.

- **Model**: [A4](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A4) \[Rezolvat\] (pentru momente), cursul despre riscul de estimare

- **Ipoteze**: randamente i.i.d. $N(\boldsymbol\mu, \boldsymbol\Sigma)$; $\hat{\boldsymbol\mu}$, $\hat{\boldsymbol\Sigma}$ de verosimilitate maximă (împărțire la $T$); se dau: $\hat{\boldsymbol\mu} \sim N(\boldsymbol\mu, \boldsymbol\Sigma/T)$, $\mathbb{E}[\hat{\boldsymbol\Sigma}^{-1}] = \frac{T}{T-N-2}\boldsymbol\Sigma^{-1}$ pentru $T > N + 2$, $\hat{\boldsymbol\mu}$ independent de $\hat{\boldsymbol\Sigma}$ și $\mathbb{E}[\mathbf{x}^\top\mathbf{A}\mathbf{x}] = \operatorname{tr}(\mathbf{A}\operatorname{Var}(\mathbf{x})) + \mathbb{E}[\mathbf{x}]^\top\mathbf{A}\,\mathbb{E}[\mathbf{x}]$

- **Cerințe**

  1.  Calculați $\mathbb{E}[\hat\theta^2 \mid \hat{\boldsymbol\mu}]$ pentru $\hat\theta^2 = \hat{\boldsymbol\mu}^\top\hat{\boldsymbol\Sigma}^{-1}\hat{\boldsymbol\mu}$.

  2.  Luați media după $\hat{\boldsymbol\mu}$ și deduceți $\mathbb{E}[\hat\theta^2] = (T\theta^2 + N)/(T-N-2)$ ([Kan & Zhou (2007)](https://doi.org/10.1017/S0022109000004129)).

  3.  Evaluați pentru $N = 9$, $T = 60$, $\theta = 0{,}66/\sqrt{12}$ lunar; raportați $\sqrt{12\,\mathbb{E}[\hat\theta^2]}$, rădăcina mediei pătratice anualizate a estimației (nu media ei), și comparați cu o simulare.

  4.  Dați estimatorul nedeplasat al lui $\theta^2$ implicat de formulă.

[Exercițiul A8 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A8)

In [12]:
# Codul vostru aici


### A9. Ponderile tangente prin regresie [Propus]

- **Model**: [A1](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A1) \[Rezolvat\] (algebră matriceală); anexa cursului despre [Britten-Jones (1999)](https://doi.org/10.1111/0022-1082.00120)

- **Cadru**: $\mathbf{X}$ este matricea $T \times N$ a randamentelor în exces, cu rîndurile $\mathbf{r}_t^\top$; $\mathbf{1}_T$ variabila dependentă (o coloană de 1), $\mathbf{1}_N$ vectorul de normalizare; $\hat{\boldsymbol\mu} = \mathbf{X}^\top\mathbf{1}_T/T$, $\hat{\boldsymbol\Sigma} = \mathbf{X}^\top\mathbf{X}/T - \hat{\boldsymbol\mu}\hat{\boldsymbol\mu}^\top$, inversabilă

- **Cerințe**

  1.  Arătați că $\mathbf{X}^\top\mathbf{X}/T = \hat{\boldsymbol\Sigma} + \hat{\boldsymbol\mu}\hat{\boldsymbol\mu}^\top$ și scrieți estimatorul OLS (ordinary least squares) al regresiei $1 = \mathbf{b}^\top\mathbf{r}_t + u_t$ (fără termen liber) în funcție de $\hat{\boldsymbol\mu}$, $\hat{\boldsymbol\Sigma}$.

  2.  Aplicați Sherman--Morrison, $(\mathbf{A} + \mathbf{u}\mathbf{u}^\top)^{-1} = \mathbf{A}^{-1} - \mathbf{A}^{-1}\mathbf{u}\mathbf{u}^\top\mathbf{A}^{-1}/(1 + \mathbf{u}^\top\mathbf{A}^{-1}\mathbf{u})$, pentru a arăta că $\hat{\mathbf{b}} = \hat{\boldsymbol\Sigma}^{-1}\hat{\boldsymbol\mu}/(1 + \hat\theta^2)$.

  3.  Arătați că $\hat{\mathbf{b}}/\mathbf{1}_N^\top\hat{\mathbf{b}}$ sînt ponderile tangente de eșantion cînd $\mathbf{1}_N^\top\hat{\boldsymbol\Sigma}^{-1}\hat{\boldsymbol\mu} > 0$.

  4.  Opțional: pentru $N = 3$ scrieți matricea de restricții a ipotezei „tangent $=$ 1/N", apoi generalizați.

[Exercițiul A9 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A9)

In [13]:
# Codul vostru aici


## Partea B — backtest-uri și inferență

### B1. 1/N față de Markowitz pe sectoare [Rezolvat]

- Interpretare: Mean–Variance, long-only (MV-LO) depășește 1/N cu 0,06, cu $p = 0{,}60$; cît de lung ar trebui să fie eșantionul pentru ca diferența să fie semnificativă?

In [14]:
SEM_BLOCKS = {'MV': 6, 'MV-LO': 1, 'GMV': 1, 'HRP-GMV': 1}   # lungimile calibrate ale blocurilor (Algoritmul 3.1)
# Rezolvare
ret, to, W = bts['Sectors']
for s in ['MV', 'MV-LO', 'GMV']:
    d, se, p = sr_diff_hac(ret[s], ret['1/N'])
    rb, _ = sr_diff_boot(ret[s], ret['1/N'], block=SEM_BLOCKS[s])
    print(f"{s:6s} Delta = {d:+.3f}  se = {se:.3f}  p HAC = {p:.3f}  interval bootstrap = {np.round(rb['ci'], 3)}  p boot = {rb['p_boot']:.3f}  block = {rb['block']}")
summary(ret, to).loc[['1/N', 'MV', 'MV-LO', 'GMV']].rename(columns=SUMCOLS).round(3)

MV     Delta = -0.404  se = 0.331  p HAC = 0.223  interval bootstrap = [-1.175  0.367]  p boot = 0.296  block = 6
MV-LO  Delta = +0.063  se = 0.120  p HAC = 0.597  interval bootstrap = [-0.187  0.314]  p boot = 0.627  block = 1


GMV    Delta = +0.050  se = 0.159  p HAC = 0.754  interval bootstrap = [-0.269  0.369]  p boot = 0.762  block = 1


,Mean,Volatility,Sharpe,Turnover,Max drawdown,Sharpe net 10 bp,Sharpe net 50 bp
1/N,0.094,0.143,0.661,0.024,-0.491,0.659,0.651
MV,0.178,0.690,0.258,3.855,-1.000,0.194,-0.080
MV-LO,0.107,0.147,0.724,0.235,-0.423,0.705,0.629
GMV,0.085,0.119,0.711,0.234,-0.326,0.687,0.594


### B2. Costul-prag [Propus]

- Replicare: calcul/cod reproductibil, comparație cu reperul din curs și interpretare.

- **Model**: [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B1) \[Rezolvat\]

- **Întrebare**: MV-LO depășește 1/N înainte de costuri pe 16 ETF-uri; la ce cost de tranzacționare dispare avantajul?

- **Date**: universul combinat (16 ETF-uri), în afara eșantionului mai 2012 -- iulie 2026 ($T = 171$); strategia 1 = MV-LO, strategia 2 = 1/N; turnover ca în [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B1)

- Randamentul net $r_t - c\,TO_t$, costul $c$ pe unitate tranzacționată: [[Costurile de tranzacționare și costul-prag]{style="color: MainBlue"}](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=sem4p11)

- **Cerințe**

  1.  Calculați rapoartele Sharpe brute, p-value-ul Heteroskedasticity and Autocorrelation Consistent (standard errors) (HAC) a diferenței lor și turnover-urile medii.

  2.  Pe o grilă de 0--200 bp (pas 0,1 bp) recalculați media **și** abaterea standard a ambelor serii nete și găsiți costul $c^*$ la care rapoartele Sharpe nete se intersectează; reprezentați ambele curbe.

  3.  Deduceți aproximarea cu volatilitate fixă $c^* \approx \Delta SR / [12(TO_1/\sigma_1 - TO_2/\sigma_2)]$ ($\sigma$ anual) și comparați-o cu valoarea de pe grilă.

- **Interpretare**: la costurile unui investitor individual pe BVB sau pe ETF-uri americane, este justificată implementarea MV-LO?

[Exercițiul B2 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B2)

In [15]:
# Codul vostru aici


### B5. Black–Litterman pe un set multi-active [Propus]

- **Model**: [A6](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A6) și [A7](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A7)

- **Date de intrare**: SPY, EFA, EEM, TLT, IEF, iShares iBoxx Investment Grade Corporate Bond ETF (LQD), iShares iBoxx High Yield Corporate Bond ETF (HYG), GLD (în această ordine), randamente lunare în exces august 2016 -- iulie 2026 ($T = 120$), $\boldsymbol\Sigma$ anualizată; reper 1/N, $\delta_{\text{RA}} = 2{,}5$, $\tau = 0{,}05$; opinia GLD minus TLT $= 4\%$ pe an, $\mathbf{P} = (0, 0, 0, -1, 0, 0, 0, +1)$, $\Omega = \tau\mathbf{P}\boldsymbol\Sigma\mathbf{P}^\top$

- Alocare statică fără restricții: o pondere negativă este o poziție short

- **Cerințe**

  1.  Calculați $\boldsymbol\pi = \delta_{\text{RA}}\boldsymbol\Sigma\mathbf{w}_b$ și diferența implicită GLD minus TLT.

  2.  Calculați varianța spread-ului, surpriza $q - \mathbf{P}\boldsymbol\pi$, scalarul de ajustare din [A7](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A7) și noile ponderi; reprezentați-le alături de reper.

  3.  Comparați cu media de eșantion a lui GLD minus TLT, cu o încredere de zece ori mai mică ($\Omega \times 10$) și cu alocarea long-only care maximizează $\boldsymbol\mu_{BL}^\top w-\delta_{\mathrm{RA}}w^\top\boldsymbol\Sigma w/2$, cu ponderi nenegative de sumă unu. Comparați media cu opinia declarată, fără să înlocuiți opinia cu ea.

- **Interpretare**: de ce se modifică ponderile atît de mult, deși opinia este modestă?

[Exercițiul B5 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B5)

In [16]:
# Codul vostru aici


### B6. Testul egalității dintre portofoliul tangent și 1/N [Propus]

- Replicare: calcul/cod reproductibil, comparație cu reperul din curs și interpretare.

- **Model**: [A9](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A9) și [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B1)

- **Date**: nouă sectoare, randamente lunare în exces ianuarie 1999 -- iulie 2026 ($T = 331$) și cele două jumătăți, ianuarie 1999 -- septembrie 2012 și octombrie 2012 -- iulie 2026

- **Testele** pentru $H_0$: $\mathbf{R}\mathbf{b} = \mathbf{0}$, $\mathbf{R} = [\mathbf{I}_8,\ -\mathbf{1}_8]$, 8 restricții

  - Exact: $F = (\mathbf{R}\hat{\mathbf{b}})^\top[\mathbf{R}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{R}^\top]^{-1}(\mathbf{R}\hat{\mathbf{b}})/(8 s^2) \sim F(8, T - 9)$, $s^2 = \hat{\mathbf{u}}^\top\hat{\mathbf{u}}/(T - 9)$

  - Wald HAC: $W = (\mathbf{R}\hat{\mathbf{b}})^\top(\mathbf{R}\hat{\mathbf{V}}\mathbf{R}^\top)^{-1}\mathbf{R}\hat{\mathbf{b}} \to \chi^2_8$, $\hat{\mathbf{V}} = T(\mathbf{X}^\top\mathbf{X})^{-1}\hat{\boldsymbol\Omega}(\mathbf{X}^\top\mathbf{X})^{-1}$, $\hat{\boldsymbol\Omega}$ Newey--West pentru $\mathbf{r}_t\hat u_t$, Bartlett, $\lfloor 4(T/100)^{2/9}\rfloor$ laguri

- **Cerințe**

  1.  Completați un tabel $3 \times 2$ cu statistici și p-value-uri (trei eșantioane, două teste); reprezentați coeficienții cu intervalele OLS și HAC.

  2.  Simulați 2 000 de eșantioane de $T = 166$ dintr-un model Normal cu $\boldsymbol\mu \propto \boldsymbol\Sigma\mathbf{1}$ (deci $H_0$ adevărată, $\boldsymbol\Sigma$ din a doua jumătate) și raportați rata de respingere a fiecărui test la 5%. Scalați vectorul mediilor astfel încît 1/N să aibă raportul Sharpe lunar observat în a doua jumătate.

- **Interpretare**

  - cînd testele conduc la concluzii diferite, căruia îi acordați încredere?

  - înseamnă nerespingerea că portofoliul tangent este 1/N?

[Exercițiul B6 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B6)

In [17]:
# Codul vostru aici


### B7. Shrinkage pe 16 ETF-uri [Propus]

- Replicare: calcul/cod reproductibil, comparație cu reperul din curs și interpretare.

- **Model**: [B3](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B3) \[Rezolvat\]

- **Date**: universul combinat (nouă sectoare plus EFA, EEM, TLT, IEF, LQD, HYG, GLD), lunile comune mai 2007 -- iulie 2026; fereastră de 60 de luni, în afara eșantionului mai 2012 -- iulie 2026 ($T = 171$)

- **Cerințe** (rezultat: un tabel cu rîndurile GMV, Global Minimum Variance with Ledoit–Wolf covariance shrinkage (GMV-LW), Global Minimum Variance with nonlinear covariance shrinkage (GMV-NL), 1/N și coloanele volatilitate, turnover, raportul varianțelor cu interval)

  1.  Volatilitatea în afara eșantionului a GMV, GMV-LW și 1/N; raportul varianțelor GMV-LW/GMV cu bootstrap-ul pereche pe blocuri din [B3](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B3); turnover-urile medii.

  2.  $\delta$ mediu și numerele de condiționare ale matricei de eșantion și ale celei cu shrinkage pe ultima fereastră.

  3.  Extindere: shrinkage neliniar ([Ledoit & Wolf (2020)](https://doi.org/10.1214/19-AOS1921); funcția `nl_shrink(X)`), backtest-ul GMV-NL și rapoartele varianțelor sale față de GMV și GMV-LW.

- **Interpretare**: este corelația constantă o țintă rezonabilă pentru acțiuni și obligațiuni de stat împreună? Distingeți o diferență semnificativă de o simplă estimație punctuală

[Exercițiul B7 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B7)

In [18]:
# Codul vostru aici


### B8. Risk parity cu Bitcoin [Propus]

- Replicare: calcul/cod reproductibil, comparație cu reperul din curs și interpretare.

- **Model**: [B4](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B4) \[Rezolvat\]; literatură: [Petukhina, Trimborn, Härdle & Elendner (2021)](https://doi.org/10.1080/14697688.2021.1880023), [Pele et al. (2023)](https://doi.org/10.1080/1351847X.2021.1960403)

- **Date**: cele opt ETF-uri multi-active cu și fără Bitcoin (BTC/United States dollar (USD)), prețuri aliniate pe zilele comune de tranzacționare (fără weekenduri), randamente de sfîrșit de lună din august 2010; ambele universuri pe **aceleași** luni

- **Cerințe**

  1.  Descriptiv, august 2010 -- iulie 2026: volatilitatea Bitcoin și cota lui de risc în 1/N cu nouă active.

  2.  În afara eșantionului, fereastră de 60 de luni, august 2015 -- iulie 2026 ($T = 132$): backtest pentru 1/N, ERC și Hierarchical Risk Parity (HRP) cu și fără Bitcoin; rapoarte Sharpe, volatilități, drawdown-uri; testul HAC al diferenței „cu Bitcoin" minus „fără Bitcoin" pentru fiecare regulă.

  3.  Reprezentați în timp ponderea Bitcoin în ERC și HRP și cota lui de risc în 1/N.

  4.  Tratați cele trei teste ca o familie: raportați p-value-urile brute și ajustate Holm.

- **Interpretare**

  - vă așteptați ca prima istorică a Bitcoin să persiste?

  - ce efect are alegerea ex post a lui Bitcoin?

[Exercițiul B8 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B8)

In [19]:
# Codul vostru aici


### B9. HRP față de GMV pe 16 ETF-uri [Propus]

- **Model**: [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B1) \[Rezolvat\]

- **Date**: universul combinat și backtest-ul din [B7](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B7) (în afara eșantionului mai 2012 -- iulie 2026, $T = 171$)

- **Cerințe**

  1.  Raportați raportul Sharpe, volatilitatea și turnover-ul pentru HRP și GMV, brut și net de 50 bp pe unitate de turnover; reprezentați punctele brute și nete.

  2.  Testul principal pe randamentele **brute** în exces: $H_0$: $SR_{\text{HRP}} = SR_{\text{GMV}}$ cu testul HAC și cu bootstrap-ul studentizat (bloc 1), la 5% și 10%; raportați intervalul bootstrap.

  3.  Completați un tabel de decizie (test $\times$ nivel).

- **Interpretare**

  - GMV are volatilitatea mai mică, HRP raportul Sharpe mai mare: ce criteriu este potrivit pentru fiecare tip de investitor?

  - ce presupune o comparație la volatilitate egală?

[Exercițiul B9 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B9)

In [20]:
# Codul vostru aici


## După curs — B3, B4

- Seminarul 4 are loc înaintea Cursului 4: aceste cerințe folosesc instrumente introduse la curs (shrinkage-ul covarianței, ERC și HRP); lucrați-le după curs, înaintea seminarului următor.

### B3. Shrinkage-ul și riscul GMV [Rezolvat]

- Interpretare:
  - de ce este cîștigul mic cu 9 active și 60 de luni?
  - cînd ar fi mai mare?

In [21]:
# Rezolvare
vr, ci, share = var_ratio_boot(ret['GMV-LW'], ret['GMV'])
print(f"vol GMV = {ret['GMV'].std() * np.sqrt(12):.4f}  vol GMV-LW = {ret['GMV-LW'].std() * np.sqrt(12):.4f}")
print(f'raportul varianțelor = {vr:.3f}  95% = {ci.round(3)}')
print(f"turnover: GMV {to['GMV'].mean():.3f}  GMV-LW {to['GMV-LW'].mean():.3f}")
# B3 extins: banda Marchenko–Pastur pe ultima fereastră
ev = np.sort(np.linalg.eigvalsh(np.corrcoef(us_monthly(SECTORS)[1].values[-WINDOW:].T)))[::-1]
lo, hi = mp_bounds(9 / WINDOW)
print('valori proprii', ev.round(3), ' banda', round(lo, 3), round(hi, 3), ' peste', (ev > hi).sum(), ' sub', (ev < lo).sum())

vol GMV = 0.1190  vol GMV-LW = 0.1180


raportul varianțelor = 0.984  95% = [0.918 1.062]
turnover: GMV 0.234  GMV-LW 0.137


valori proprii [5.246 1.349 0.835 0.499 0.32  0.29  0.212 0.143 0.107]  banda 0.375 1.925  peste 1  sub 5


### B4. ERC și HRP pe sectoare [Rezolvat]

- Interpretare: ERC și 1/N au corelația 0,993; este ERC o strategie diferită sau doar o mică abatere de la 1/N?

In [22]:
# Rezolvare
X = us_monthly(SECTORS)[1].values[-WINDOW:]
S9 = np.cov(X.T)
print(pd.DataFrame({'ERC': w_erc(S9), 'HRP': w_hrp(S9), 'cota de risc 1/N': risk_contrib(w_ew(9), S9)},
                   index=SECTORS).round(3))
for s in ['ERC', 'HRP']:
    d, se, p = sr_diff_hac(ret[s], ret['1/N'])
    print(f"{s}: SR = {sharpe(ret[s]):.3f}  Delta = {d:+.3f}  p = {p:.3f}  corr(1/N) = {ret[[s, '1/N']].corr().iloc[0, 1]:.4f}  turnover = {to[s].mean():.3f}")

       ERC    HRP  cota de risc 1/N
XLB  0.086  0.045             0.138
XLE  0.109  0.093             0.108
XLF  0.106  0.100             0.114
XLI  0.090  0.050             0.134
XLK  0.102  0.111             0.121
XLP  0.148  0.184             0.078
XLU  0.125  0.171             0.093
XLV  0.139  0.152             0.083
XLY  0.095  0.094             0.131
ERC: SR = 0.699  Delta = +0.038  p = 0.091  corr(1/N) = 0.9927  turnover = 0.028
HRP: SR = 0.700  Delta = +0.039  p = 0.325  corr(1/N) = 0.9789  turnover = 0.082


## Partea C — întrebări deschise și critica unui răspuns AI

### C1. Critica unui răspuns AI [Propus]

- **Model**: [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B1) și [A4](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A4) \[Rezolvat\]. **Cererea**: „`prices` conține prețurile zilnice ajustate ale celor nouă ETF-uri sectoriale din [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B1) (decembrie 1998 -- iulie 2026). Scrie cod Python pentru un backtest GMV în afara eșantionului, cu reechilibrare lunară, și dă-mi raportul Sharpe anualizat."

``` {showstringspaces="false"}
1 import numpy as np
2 r = prices.resample('ME').last().pct_change().dropna()  # monthly simple returns
3 S = r.cov().values                                      # covariance matrix
4 w = np.linalg.solve(S, np.ones(len(S)))                 # Sigma^{-1} 1
5 w = w / w.sum()                                         # GMV weights, sum to one
6 port = r.dot(w)                                         # portfolio return
7 sr = port.mean() / port.std() * np.sqrt(252)
8 print(f"Annualised Sharpe ratio: {sr:.2f}")
```

- **Cerințe** (`rf`: rata lunară a titlului de stat din pregătire, în zecimale, aliniată pe datele de sfîrșit de lună)

  1.  Găsiți cele trei erori intenționate; numiți linia, explicați de ce este greșită și cum ați detecta-o (o verificare a datelor, un număr din [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B1) sau [A4](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A4), o definiție). Alte observații argumentate sînt binevenite.

  2.  Folosiți schema B1: estimare mobilă pe 60 de luni, ponderi alese înaintea lunii, rebalansare lunară cu drift. Scrieți codul corectat și raportați Sharpe; verificați decalajul datelor, valorile lipsă, suma ponderilor și anualizarea.

  3.  Notați fiecare eroare în formatul `AI_ERRORS.md`: cererea, răspunsul, eroarea, cum a fost găsită, corecția.

- **Interpretare**

  - cît de mult se abate valoarea raportată de AI de la una care poate fi susținută?

  - care eroare contează cel mai mult?

[Exercițiul C1 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-C1)

In [23]:
# Codul dumneavoastră corectat pentru C1


### C2. Un portofoliu BVB în RON: aduce diversificarea dincolo de BVB un cîștig? [Propus]

- **Model**: [B1](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B1), [B4](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-B4) și [A4](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-A4) \[Rezolvat\]. **Întrebare**: aduce diversificarea dincolo de BVB un cîștig unui investitor român?

- **Date și monedă**: Banca Transilvania (TLV), OMV Petrom (SNP), BRD Groupe Société Générale (BRD), Transgaz (TGN), Romgaz (SNG), Nuclearelectrica (SNN), Electrica (EL), Transelectrica (TEL) și indicele BET Total Return (index) (BET-TR) (octombrie 2014 -- august 2026); SPY, EFA, IEF, GLD convertite în lei în zilele comune, $P_{\text{RON}} = P_{\text{USD}} \times$ USD/RON ([Banca Națională a României (BNR)](https://www.bnr.ro)); de exemplu 2017-10-02: SPY 219,84 USD $\times$ 3,9086 $=$ 859,27 lei

- **Cazul de bază**: randamente lunare în lei, fereastră de 36 de luni, primul portofoliu în octombrie 2017, $R_f = 0$

- **Cerințe**

  1.  Realizați backtesting-ul pentru 1/N, GMV-LW, ERC și HRP pe cele opt acțiuni, apoi pe acțiuni plus cele patru ETF-uri străine.

  2.  Comparați fiecare portofoliu cu BET-TR (niciodată cu Bucharest Exchange Trading (index) (BET)): raport Sharpe, volatilitate, drawdown maxim; reprezentați averea și drawdown-ul.

  3.  Testați fiecare regulă globală față de BET-TR și față de varianta ei doar cu BVB (HAC), cu ajustare Holm pe această familie de opt teste.

  4.  Sensibilitate: o rată în lei de 5% pe an, $0{,}05/12$ pe lună, scăzută din toate randamentele.

- **Livrabile**: un folder Quantlet în propriul repository, un tabel cu cele opt teste ($p$ brut și Holm), un slide cu concluzia și principala ei limită (riscul valutar, costuri, impozite)

[Exercițiul C2 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-C2)

In [24]:
# Codul vostru aici


### C3. Frontiera implementabilă pe datele cursului [Propus]

- **Întrebare**: se păstrează ierarhia regulilor care țin cont de costuri din [Jensen, Kelly, Malamud & Pedersen (2026)](https://doi.org/10.1093/rfs/hhag022) (Tabelul 2; studiul de caz din curs) pe un univers mic?

- **Un proiect de cercetare**, alternativă la [C2](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-C2); cod de pornire: `jkmp_replication` din Quantlet-ul `MFM_ch4_implementable_frontier`

- **Notație**

  - $\pi_t$: ponderile riscante; tranzacția $\pi_t - g_t\pi_{t-1}$, cu $g_t$ drift-ul ponderilor din luna trecută; costul $\tfrac{W_t}{2}(\pi_t - g_t\pi_{t-1})^\top\boldsymbol\Lambda(\pi_t - g_t\pi_{t-1})$, $\Lambda_i = 0{,}2/V_i$, $V_i$ volumul zilnic mediu în dolari pe 6 luni

  - Utilitatea lunară: randamentul net în exces $- \tfrac{\gamma}{2}\times$ varianța lui, $\gamma = 10$; regula din articol parcurge prin tranzacții doar o parte ($m$) din distanța pînă la un portofoliu-țintă învățat din semnale

- **Date**: JPMorgan Chase & Co. (NYSE ticker) (JPM), Bank of America Corporation (NYSE ticker) (BAC), C (Citigroup), The Goldman Sachs Group, Inc. (NYSE ticker) (GS), Morgan Stanley (NYSE ticker) (MS), Wells Fargo & Company (NYSE ticker) (WFC), NVIDIA Corporation (Nasdaq ticker) (NVDA), Apple Inc. (Nasdaq ticker) (AAPL), Microsoft Corporation (Nasdaq ticker) (MSFT), Amazon.com, Inc. (Nasdaq ticker) (AMZN), Alphabet Inc., Class A shares (Nasdaq ticker) (GOOGL), GameStop Corp. (NYSE ticker) (GME), Cisco Systems, Inc. (Nasdaq ticker) (CSCO), Strategy Inc., formerly MicroStrategy (Nasdaq ticker) (MSTR) și cele nouă ETF-uri sectoriale, prețuri aliniate pe zilele comune (din 19 august 2004); titlul de stat din pregătire

- Semnalele la fiecare sfîrșit de lună, ordonate în $[0, 1]$: $\ln(P_t/P_{t-1})$, $\ln(P_{t-1}/P_{t-6})$, $\ln(P_{t-1}/P_{t-12})$, volatilitatea pe un an a randamentelor zilnice, logaritmul volumului în dolari pe 6 luni

- **Calendar**: semnalele și covarianțele folosesc doar date trecute; ferestrele de covarianță cresc din august 2004 pînă ajung la 10 ani (august 2014); hiperparametrii se aleg anual doar pe anii de validare de după 2011; primele ponderi de test la sfîrșitul lui ianuarie 2014, randamentele de test februarie 2014 -- august 2026

- **Cerințe**

  1.  Întîi un caz de bază: construiți Portfolio-ML cu alegerea fixă de pornire $\lambda=1$, $p=64$, $\eta=e^{-3}$ (cerința următoare calibrează grila); verificați pas cu pas costul unei luni ($W_t$, vectorul tranzacțiilor, $\boldsymbol\Lambda$).

  2.  Calibrați pe grila din articol (Tabelul 1) și raportați, pentru Portfolio-ML, 1/N, varianță minimă, Markowitz-ML și Static-ML: randament, volatilitate, Sharpe brut și net, cost, utilitate, turnover, levier (formatul Tabelului 2).

  3.  Probabilitatea a posteriori a unei utilități mai mari, $\Phi(\bar d/(s_d/\sqrt{T}))$ pentru diferențele lunare de utilitate $d_t$ (distribuția Normală, a priori plată, dependența serială ignorată).

  4.  Extindere: avere de 1, 10, 100 de miliarde de dolari $\times$ $\gamma \in \{1, 5, 10, 20, 100\}$; reprezentați frontierele implementabile, cu costurile lui 1/N calculate la fiecare nivel de avere.

- Utilitatea lunară: $u_t=r_t^{net}-\frac{\gamma}{2}(r_t^{net}-\bar r^{net})^2$, cu media întregii perioade evaluate; $d_t=u_{1t}-u_{2t}$. Înmulțirea tuturor fluxurilor cu 12 nu schimbă probabilitatea afișată.

- **Raportați**: două tabele, figura frontierelor, un folder Quantlet; distingeți elementele preluate din articol de simplificările introduse

- **Interpretare**

  - dacă Portfolio-ML are un raport Sharpe net mai mic decît 1/N, dar o utilitate mai mare, ce criteriu se potrivește investitorului cu $\gamma = 10$?

  - pentru ce $\gamma$ s-ar inversa ierarhia?

[Exercițiul C3 în seminar](https://danpele.github.io/MFM/RO/Seminarii/seminar4_optimizarea_portofoliului_ro.pdf#nameddest=ex-C3)

In [25]:
# Codul dumneavoastră pentru C3
